In [2]:
# %%
# Cell 1
# Simplified Background Analysis
# ROI around Q_ββ — Bi-214 and Tl-208 in EFCu (Reentrant tube)
# Imports
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os

plt.rcParams["font.size"] = 14

In [3]:
# %%
# Cell 2
# Configuration — all analysis parameters in one place

# --- Paths and files ---
base_path = "./"

# Switch between file sets here. optical_map files have no SiPM info
# (npe unavailable), full_sipm files have real npe.
isotope_files = {
    # full_sipm set (5e7 primaries per file, has npe)
    #"Bi214": "Bi214_EFCu_vtx.parquet",
    #"Tl208": "Tl208_EFCu_vtx.parquet",

    # optical_map set (1e9 primaries per file, no npe -> falls back to LAr energy veto)
    "Bi214": "Bi214_EFCu_RT_nolayer_optical_map_vtx.parquet",
    "Tl208": "Tl208_EFCu_RT_nolayer_optical_map_vtx.parquet",
}

# Total simulated primaries thrown per file.
# Set per-isotope in case files from different production runs are mixed.
n_sim_total = {
    "Bi214": 1e9,
    "Tl208": 1e9,
    
    #"Bi214": 5e7,
    #"Tl208": 5e7,
}

# --- Detector / normalization ---
Ge_total_mass_kg = 1000.0         # detector mass for normalization [kg]
seconds_per_year = 365.25 * 24 * 3600

# --- ROI and cuts ---
Q_bb              = 2039   # keV
window_half_width = 55     # keV
ROI_low           = Q_bb - window_half_width
ROI_high          = Q_bb + window_half_width

npe_threshold        = 4      # photoelectrons from SiPMs (AC UAr veto)
lar_energy_threshold_kev = 20 # keV, used when npe is unavailable
aoe_class_threshold  = -1.80  # from aoe_class_paras.yaml (PSD)
m1_energy_threshold  = 5      # keV (M1 single-hit requirement)

# --- Spectrum binning ---
spec_e_min     = 1000     # keV
spec_e_max     = 3000  # keV
spec_bin_width = 10    # keV
spec_bins      = np.arange(spec_e_min, spec_e_max + spec_bin_width, spec_bin_width)

# --- Material geometry ---
# Materials ordered top to bottom, with the THICKNESS of each part except
# the last (which extends to the observed bottom of the vertex distribution).
material_order       = ["steel", "Cu", "EFCu"]
boundary_distances_m = [2.062, 4.067]   # cumulative distances from top to each boundary [steel/Cu, Cu/EFCu], meters

density_lookup = {
    "steel": 7900.0,   # kg/m^3
    "Cu":    8960.0,   # kg/m^3, regular copper
    "EFCu":  8930.0,   # kg/m^3, electroformed copper
}

voxel_size = 0.005  # meters, for volume estimation via voxel occupancy

# --- Specific activities (µBq/kg -> Bq/kg) ---
# Bi214 belongs to the 238U chain, Tl208 to the 232Th chain.
specific_activity_bq_per_kg = {
    "Bi214": {"steel": 2500e-6, "Cu": 1e-6, "EFCu": 0.19e-6}, #U238
    "Tl208": {"steel": 1000e-6, "Cu": 1e-6, "EFCu": 0.37e-6}, #Th232
}

# --- Print summary ---
print(f"Q_ββ = {Q_bb} keV,  ROI = [{ROI_low}, {ROI_high}] keV")
print(f"Spectrum: {spec_e_min}–{spec_e_max} keV, {spec_bin_width} keV bins ({len(spec_bins)-1} bins)")
print(f"AC UAr   : NPE ≤ {npe_threshold}  (fallback: LAr energy ≤ {lar_energy_threshold_kev} keV)")
print(f"PSD      : AoE_class > {aoe_class_threshold}")
print(f"M1       : exactly 1 Ge hit with energy > {m1_energy_threshold} keV")
print(f"Geometry : {material_order}, boundaries {boundary_distances_m} m")
print(f"N_sim    : { {k: f'{v:.2e}' for k, v in n_sim_total.items()} }")

Q_ββ = 2039 keV,  ROI = [1984, 2094] keV
Spectrum: 1000–3000 keV, 10 keV bins (200 bins)
AC UAr   : NPE ≤ 4  (fallback: LAr energy ≤ 20 keV)
PSD      : AoE_class > -1.8
M1       : exactly 1 Ge hit with energy > 5 keV
Geometry : ['steel', 'Cu', 'EFCu'], boundaries [2.062, 4.067] m
N_sim    : {'Bi214': '1.00e+09', 'Tl208': '1.00e+09'}


In [4]:
# %%
# Cell 3
# Load data, detect npe availability, print field descriptions
def npe_is_available(sim):
    """Check whether sim.npe holds real numeric values or is an
    all-null/unknown-type placeholder column."""
    type_str = str(ak.type(sim.npe)).lower()
    return "unknown" not in type_str and "none" not in type_str


def describe_fields(sim, name=""):
    """Print every field with its awkward type and a preview of first 3 events."""
    print(f"=== Fields in {name} ({len(sim):,} events) ===")
    width = max(len(f) for f in sim.fields)
    for field in sim.fields:
        sample = sim[field][:5].tolist()
        print(f"  {field:<{width}} : {ak.type(sim[field])}")
        print(f"  {'':<{width}}   first 5: {sample}")
    print()


sims = {}
has_npe = {}
for iso, fname in isotope_files.items():
    path = f"{base_path}/{fname}"
    sims[iso] = ak.from_parquet(path)
    has_npe[iso] = npe_is_available(sims[iso])
    print(f"{iso}: {len(sims[iso]):,} events loaded from {fname} (npe available: {has_npe[iso]})")

output_dir = f"{base_path}/plots"
os.makedirs(output_dir, exist_ok=True)

for iso, sim in sims.items():
    describe_fields(sim, name=iso)

ValueError: no *.parquet or *.parq matches for path './/Bi214_EFCu_RT_nolayer_optical_map_vtx.parquet'

In [5]:
# %%
# Cell 4
# NPE vs LAr energy and NPE vs ALAr energy (per isotope)
# Skipped automatically for files where npe is unavailable.
def plot_npe_vs_energy(npe_total, energy_total, iso, energy_label, fname_suffix, npe_max=50):
    npe_np    = ak.to_numpy(npe_total)
    energy_np = ak.to_numpy(energy_total)

    fig, ax = plt.subplots(figsize=(8, 7))
    h = ax.hist2d(energy_np, npe_np, bins=50,
                  range=[[energy_np.min(), 500], [0, 50]],
                  norm=plt.matplotlib.colors.LogNorm(), cmap='viridis')
    plt.colorbar(h[3], ax=ax, label='Counts')
    ax.axhline(npe_threshold, color='red', linestyle='--', linewidth=1.5,
               label=f'NPE threshold = {npe_threshold}')
    ax.set_xlabel(f'{energy_label} [keV]')
    ax.set_ylabel('NPE')
    ax.set_ylim(0, npe_max)
    ax.legend(fontsize=11)
    ax.set_title(f'NPE vs {energy_label} — {iso} (zoomed, NPE 0–{npe_max})')
    plt.grid()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/npe_vs_{fname_suffix}_{iso}_zoom.png", dpi=150)
    plt.show()


for iso, sim in sims.items():
    if not has_npe[iso]:
        print(f"Skipping NPE plots for {iso}: npe field unavailable in this file.")
        continue
    total_npe  = ak.sum(sim.npe, axis=-1)
    total_lar  = ak.sum(sim.lar_energy, axis=-1)
    total_alar = ak.sum(sim.alar_energy, axis=-1)
    plot_npe_vs_energy(total_npe, total_lar,  iso, 'LAr energy',  'lar_energy')
    plot_npe_vs_energy(total_npe, total_alar, iso, 'ALAr energy', 'alar_energy')

In [6]:
# %%
# Cell 5
# Build cut masks (per isotope)
def build_masks(sim, has_npe):
    psd_pass = sim.AoE_class > aoe_class_threshold

    if has_npe:
        uar_veto_pass = ak.sum(sim.npe, axis=-1) <= npe_threshold
    else:
        uar_veto_pass = ak.sum(sim.lar_energy, axis=-1) <= lar_energy_threshold_kev

    m1_pass     = ak.sum(sim.energy > m1_energy_threshold, axis=-1) == 1
    m1_uar_pass = m1_pass & uar_veto_pass

    return {
        "psd_pass":      psd_pass,
        "uar_veto_pass": uar_veto_pass,
        "m1_pass":       m1_pass,
        "m1_uar_pass":   m1_uar_pass,
    }


masks_by_isotope = {iso: build_masks(sim, has_npe[iso]) for iso, sim in sims.items()}

for iso, sim in sims.items():
    m = masks_by_isotope[iso]
    veto_method = "NPE" if has_npe[iso] else f"LAr energy ≤ {lar_energy_threshold_kev} keV"
    print(f"--- {iso} (AC UAr veto via {veto_method}) ---")
    print(f"  Total events               : {len(sim):,}")
    print(f"  Events passing M1          : {ak.sum(m['m1_pass']):,}")
    print(f"  Events passing AC UAr      : {ak.sum(m['uar_veto_pass']):,}")
    print(f"  Events passing M1 + AC UAr : {ak.sum(m['m1_uar_pass']):,}")

In [ ]:
# %%
# Cell 6
# Helper functions for ROI counting
def count_hits_in_roi(energies, roi_low, roi_high, mask=None):
    if mask is not None:
        energies = energies[mask]
    flat = ak.flatten(energies)
    return int(ak.sum((flat >= roi_low) & (flat <= roi_high)))


def count_events_in_roi(energies, roi_low, roi_high, mask=None):
    if mask is not None:
        energies = energies[mask]
    return int(ak.sum(ak.any((energies >= roi_low) & (energies <= roi_high), axis=-1)))

In [7]:
# %%
# Cell 7
# Ge spectra with progressive cuts (per isotope), raw counts
spectra_by_isotope = {}

for iso, sim in sims.items():
    m = masks_by_isotope[iso]
    psd_pass    = m["psd_pass"]
    m1_pass     = m["m1_pass"]
    m1_uar_pass = m["m1_uar_pass"]

    ge_m1         = ak.flatten(sim.energy[m1_pass])
    ge_m1         = ge_m1[ge_m1 > 0]

    ge_m1_uar     = ak.flatten(sim.energy[m1_uar_pass])
    ge_m1_uar     = ge_m1_uar[ge_m1_uar > 0]

    psd_m1_uar    = psd_pass[m1_uar_pass]
    ge_m1_uar_psd = ak.flatten(sim.energy[m1_uar_pass][psd_m1_uar])
    ge_m1_uar_psd = ge_m1_uar_psd[ge_m1_uar_psd > 0]

    spectra_by_isotope[iso] = {
        "ge_m1":         ge_m1,
        "ge_m1_uar":     ge_m1_uar,
        "ge_m1_uar_psd": ge_m1_uar_psd,
        "psd_m1_uar":    psd_m1_uar,
    }

    veto_label = f"NPE ≤ {npe_threshold}" if has_npe[iso] else f"LAr energy ≤ {lar_energy_threshold_kev} keV"

    fig, ax = plt.subplots(figsize=(10, 7))
    for data, label, color in [
        (ge_m1,         'M1',                                                     'C0'),
        (ge_m1_uar,     f'M1 + AC UAr ({veto_label})',                            'C1'),
        (ge_m1_uar_psd, f'M1 + AC UAr + PSD (AoE_class > {aoe_class_threshold})', 'C3'),
    ]:
        if len(data) > 0:
            ax.hist(ak.to_numpy(data), bins=spec_bins,
                    histtype='step', linewidth=2, label=label, color=color, alpha=0.85)

    ax.axvline(ROI_low,  color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(ROI_high, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvspan(ROI_low, ROI_high, alpha=0.08, color='red',
               label=f'ROI [{ROI_low}–{ROI_high} keV]')
    ax.set_xlabel('Energy [keV]')
    ax.set_ylabel('Counts')
    ax.set_yscale('log')
    ax.set_xlim(spec_e_min, spec_e_max)
    ax.legend(fontsize=12)
    ax.set_title(f'Ge spectrum — {iso} in EFCu (Reentrant tube)')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ge_spectra_cuts_{iso}.png", dpi=150)
    plt.show()

In [8]:
# %%
# Cell 8
# Interactive 3D vertex histogram (per isotope)
def plot_vertex_histogram_3d(sim, title, nbins=100):
    vtx_x = ak.to_numpy(sim.vtx_x)
    vtx_y = ak.to_numpy(sim.vtx_y)
    vtx_z = ak.to_numpy(sim.vtx_z)
    valid = np.isfinite(vtx_x) & np.isfinite(vtx_y) & np.isfinite(vtx_z)
    vtx_x, vtx_y, vtx_z = vtx_x[valid], vtx_y[valid], vtx_z[valid]

    H, edges = np.histogramdd(np.column_stack([vtx_x, vtx_y, vtx_z]), bins=nbins)
    x_centers = 0.5 * (edges[0][:-1] + edges[0][1:])
    y_centers = 0.5 * (edges[1][:-1] + edges[1][1:])
    z_centers = 0.5 * (edges[2][:-1] + edges[2][1:])
    xc, yc, zc = np.meshgrid(x_centers, y_centers, z_centers, indexing="ij")
    counts = H.flatten()
    mask = counts > 0

    fig = go.Figure(data=go.Scatter3d(
        x=xc.flatten()[mask], y=yc.flatten()[mask], z=zc.flatten()[mask],
        mode="markers",
        marker=dict(size=4, color=counts[mask], colorscale="Viridis",
                    colorbar=dict(title="counts"), opacity=0.8),
    ))
    fig.update_layout(
        scene=dict(xaxis_title="vtx_x", yaxis_title="vtx_y", zaxis_title="vtx_z"),
        title=title, width=800, height=700,
    )
    fig.show()


for iso, sim in sims.items():
    plot_vertex_histogram_3d(sim, title=f"Vertex position histogram (interactive) — {iso} EFCu")

In [9]:

# %%
# Cell 9
# Min/max vtx_z per isotope and pooled. Pooled data used for volume
# estimation (same physical geometry for both isotopes).
valid_vtx = {}
 
for iso, sim in sims.items():
    vtx_x = ak.to_numpy(sim.vtx_x)
    vtx_y = ak.to_numpy(sim.vtx_y)
    vtx_z = ak.to_numpy(sim.vtx_z)
    valid = np.isfinite(vtx_x) & np.isfinite(vtx_y) & np.isfinite(vtx_z)
    vtx_x, vtx_y, vtx_z = vtx_x[valid], vtx_y[valid], vtx_z[valid]
    valid_vtx[iso] = {"x": vtx_x, "y": vtx_y, "z": vtx_z}
    print(f"{iso}: vtx_z min = {vtx_z.min():.4f} m, max = {vtx_z.max():.4f} m")
 
pooled_vtx = {
    "x": np.concatenate([v["x"] for v in valid_vtx.values()]),
    "y": np.concatenate([v["y"] for v in valid_vtx.values()]),
    "z": np.concatenate([v["z"] for v in valid_vtx.values()]),
}
print(f"pooled: vtx_z min = {pooled_vtx['z'].min():.4f} m, "
      f"max = {pooled_vtx['z'].max():.4f} m, n = {len(pooled_vtx['z']):,}")
 
# r_inner and r_outer as a function of z (every 10 cm)
z_slice_width = 0.005  # meters (1 cm)
z_edges   = np.arange(pooled_vtx["z"].min(), pooled_vtx["z"].max() + z_slice_width, z_slice_width)
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
r_all     = np.sqrt(pooled_vtx["x"]**2 + pooled_vtx["y"]**2)
 
r_inner = np.full(len(z_centers), np.nan)
r_outer = np.full(len(z_centers), np.nan)
 
for k, (z_lo, z_hi) in enumerate(zip(z_edges[:-1], z_edges[1:])):
    in_slice = (pooled_vtx["z"] >= z_lo) & (pooled_vtx["z"] < z_hi)
    r_slice  = r_all[in_slice]
    if len(r_slice) > 0:
        r_inner[k] = r_slice.min()
        r_outer[k] = r_slice.max()
 
wall_thickness = (r_outer - r_inner) * 100  # cm
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10), sharey=True)
 
# Left: r_inner and r_outer
ax1.plot(r_inner * 100, z_centers, label="r_inner", color="C0", linewidth=1.0)
ax1.plot(r_outer * 100, z_centers, label="r_outer", color="C1", linewidth=1.0)
ax1.fill_betweenx(z_centers, r_inner * 100, r_outer * 100, alpha=0.15, color="C0")
ax1.set_xlabel("r [cm]")
ax1.set_ylabel("vtx_z [m]")
ax1.set_title("Inner / outer radius vs z")
ax1.legend(fontsize=10)
ax1.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
 
# Right: wall thickness
ax2.plot(wall_thickness, z_centers, color="C2", linewidth=1.0)
ax2.fill_betweenx(z_centers, 0, wall_thickness, alpha=0.15, color="C2")
ax2.set_xlabel("Wall thickness [cm]")
ax2.set_title("Wall thickness vs z")
ax2.set_xscale('log')
ax2.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
 
# Material boundaries on both panels
_top_z = pooled_vtx["z"].max()
for i, (dist, label) in enumerate(zip(
    boundary_distances_m,
    [f"{material_order[0]}/{material_order[1]}",
     f"{material_order[1]}/{material_order[2]}"]
)):
    z_boundary = _top_z - dist
    for ax in (ax1, ax2):
        ax.axhline(z_boundary, color=f"C{i+3}", linestyle='--',
                   linewidth=1.2, label=label)
 
ax1.legend(fontsize=9)
ax2.legend(fontsize=9)
 
plt.tight_layout()
plt.savefig(f"{output_dir}/r_profile_vs_z.png", dpi=150)
plt.show()
 


ValueError: need at least one array to concatenate

In [10]:
# %%
# Cell 10
# Volume per material region from pooled vertex data
def build_contiguous_regions(material_order, boundary_distances_m, bottom_distance_m):
    """Build (material, dist_start, dist_end) tuples from cumulative distances
    from the top to each material boundary. Values are used directly as edges,
    NOT summed: [2.062, 4.067] means first boundary at 2.062 m from top,
    second at 4.067 m from top."""
    edges = [0.0] + list(boundary_distances_m) + [bottom_distance_m]
    return [(mat, edges[i], edges[i + 1]) for i, mat in enumerate(material_order)]


def get_region_z_bounds(top_z, dist_start, dist_end):
    return top_z - dist_end, top_z - dist_start


def estimate_volume_voxel(x, y, z, voxel_size):
    bins_x = np.arange(x.min(), x.max() + voxel_size, voxel_size)
    bins_y = np.arange(y.min(), y.max() + voxel_size, voxel_size)
    bins_z = np.arange(z.min(), z.max() + voxel_size, voxel_size)
    H, _ = np.histogramdd(np.column_stack([x, y, z]), bins=[bins_x, bins_y, bins_z])
    return np.sum(H > 0) * voxel_size**3


vtx_x, vtx_y, vtx_z = pooled_vtx["x"], pooled_vtx["y"], pooled_vtx["z"]
top_z = vtx_z.max()  # override here if you have an exact CAD reference
bottom_distance_m = top_z - vtx_z.min()

material_regions = build_contiguous_regions(material_order, boundary_distances_m, bottom_distance_m)

region_info = {}
print(f"top_z = {top_z:.4f} m")
for material, dist_start, dist_end in material_regions:
    z_low, z_high = get_region_z_bounds(top_z, dist_start, dist_end)
    in_region = (vtx_z >= z_low) & (vtx_z < z_high)
    n_in = int(in_region.sum())
    if n_in == 0:
        print(f"  {material:<6}: no vertices in z [{z_low:.4f}, {z_high:.4f})")
        continue
    volume_m3 = estimate_volume_voxel(vtx_x[in_region], vtx_y[in_region], vtx_z[in_region], voxel_size)
    region_info[material] = {"z_low": z_low, "z_high": z_high,
                              "n_vertices": n_in, "volume_m3": volume_m3}
    print(f"  {material:<6}: z [{z_low:.4f}, {z_high:.4f})  "
          f"n={n_in:,}  volume={volume_m3*1e6:.2f} cm^3")

NameError: name 'pooled_vtx' is not defined

In [11]:

# %%
# Cell 11
# 2D cross-sections of the vertex distribution from the pooled data.
# Side view (x vs z): shows the tube profile and material boundaries.
# Radial view (r vs z): collapses phi, cleaner for cylindrical geometry.
# Top-down view (x vs y): shows the circular cross-section.
r_pooled = np.sqrt(pooled_vtx["x"]**2 + pooled_vtx["y"]**2)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
 
# Side view: x vs z
axes[0].hist2d(pooled_vtx["x"], pooled_vtx["z"], bins=100,
               norm=plt.matplotlib.colors.LogNorm(), cmap="viridis")
axes[0].set_xlabel("vtx_x [m]")
axes[0].set_ylabel("vtx_z [m]")
axes[0].set_title("Side view (x vs z)")
 
# Radial view: r vs z
axes[1].hist2d(r_pooled, pooled_vtx["z"], bins=100,
               norm=plt.matplotlib.colors.LogNorm(), cmap="viridis")
axes[1].set_xlabel("r = √(x²+y²) [m]")
axes[1].set_ylabel("vtx_z [m]")
axes[1].set_title("Radial view (r vs z)")
 
# Top-down view: x vs y
axes[2].hist2d(pooled_vtx["x"], pooled_vtx["y"], bins=100,
               norm=plt.matplotlib.colors.LogNorm(), cmap="viridis")
axes[2].set_xlabel("vtx_x [m]")
axes[2].set_ylabel("vtx_y [m]")
axes[2].set_title("Top-down view (x vs y)")
axes[2].set_aspect("equal")
 
# Mark material boundaries as horizontal lines on the two z-axis plots
for ax in (axes[0], axes[1]):
    for i, (material, dist_start, dist_end) in enumerate(material_regions):
        z_low, z_high = get_region_z_bounds(top_z, dist_start, dist_end)
        ax.axhline(z_low,  color=f"C{i}", linestyle='--', linewidth=1.2,
                   label=f"{material} start")
        ax.axhline(z_high, color=f"C{i}", linestyle=':',  linewidth=1.0)
    ax.legend(fontsize=9)
 
plt.tight_layout()
plt.savefig(f"{output_dir}/vertex_cross_sections.png", dpi=150)
plt.show()
 


NameError: name 'pooled_vtx' is not defined

In [ ]:
# %%
# Cell 11
# Mass per material region
for material, info in region_info.items():
    mass_kg = info["volume_m3"] * density_lookup[material]
    info["mass_kg"] = mass_kg
    print(f"{material:<6}: mass = {mass_kg:.4f} kg  (density {density_lookup[material]:.1f} kg/m^3)")

In [ ]:
# %%
# Cell 12
# Activity per material region, in Bq (uses specific_activity_bq_per_kg from Cell 2)
activity_by_isotope = {}

for iso, sa_by_material in specific_activity_bq_per_kg.items():
    activity_by_isotope[iso] = {}
    print(f"--- {iso} ---")
    for material, info in region_info.items():
        sa = sa_by_material.get(material)
        if sa is None:
            print(f"  {material:<6}: specific activity not set, skipping")
            continue
        activity_bq = info["mass_kg"] * sa
        activity_by_isotope[iso][material] = activity_bq
        print(f"  {material:<6}: {activity_bq:.4e} Bq  "
              f"({info['mass_kg']:.4f} kg × {sa:.2e} Bq/kg)")

In [ ]:
# %%
# Cell 13
# Scale factor per isotope per region: cts/(yr·kg) per raw MC event.
# Denominator is the estimated number of primaries thrown into region R:
# N_total × V_R / V_total  (vertices are sampled uniformly in the total volume)
V_total = sum(info["volume_m3"] for info in region_info.values())
print(f"Total sampled volume: {V_total*1e6:.2f} cm^3\n")

scale_factor_by_isotope = {}

for iso, regions in activity_by_isotope.items():
    scale_factor_by_isotope[iso] = {}
    n_total = n_sim_total[iso]
    print(f"--- {iso}  (N_sim = {n_total:.2e}) ---")
    for material, activity_bq in regions.items():
        V_R = region_info[material]["volume_m3"]
        n_thrown_R = n_total * V_R / V_total
        decays_per_year = activity_bq * seconds_per_year
        sf = decays_per_year / n_thrown_R / Ge_total_mass_kg
        scale_factor_by_isotope[iso][material] = sf
        print(f"  {material:<6}: V_R={V_R*1e6:.2f} cm^3  "
              f"n_thrown_R={n_thrown_R:.2e}  "
              f"decays/yr={decays_per_year:.4e}  "
              f"scale factor={sf:.4e} cts/(yr·kg)")

In [ ]:
# %%
# Cell 14
# Normalized Ge spectrum: cts / keV / kg / yr (per isotope)
def assign_region(vtx_z, material_regions, top_z):
    region_name = np.full(vtx_z.shape, None, dtype=object)
    for material, dist_start, dist_end in material_regions:
        z_low, z_high = get_region_z_bounds(top_z, dist_start, dist_end)
        region_name[(vtx_z >= z_low) & (vtx_z < z_high)] = material
    return region_name


def weighted_flat_energy(energy_jagged, weight_per_event):
    weight_broadcast, _ = ak.broadcast_arrays(weight_per_event, energy_jagged)
    flat_energy = ak.flatten(energy_jagged)
    flat_weight = ak.flatten(weight_broadcast)
    positive = flat_energy > 0
    return ak.to_numpy(flat_energy[positive]), ak.to_numpy(flat_weight[positive])


for iso, sim in sims.items():
    vtx_z_iso   = ak.to_numpy(sim.vtx_z)
    region_name = assign_region(vtx_z_iso, material_regions, top_z)

    weight_per_event = np.zeros(len(sim))
    for material, sf in scale_factor_by_isotope[iso].items():
        weight_per_event[region_name == material] = sf
    weight_per_event = ak.Array(weight_per_event)

    m           = masks_by_isotope[iso]
    m1_pass     = m["m1_pass"]
    m1_uar_pass = m["m1_uar_pass"]
    psd_m1_uar  = spectra_by_isotope[iso]["psd_m1_uar"]

    e_m1, w_m1 = weighted_flat_energy(sim.energy[m1_pass], weight_per_event[m1_pass])

    energy_m1_uar          = sim.energy[m1_uar_pass]
    weight_m1_uar_per_event = weight_per_event[m1_uar_pass]
    e_m1_uar, w_m1_uar     = weighted_flat_energy(energy_m1_uar, weight_m1_uar_per_event)

    weight_m1_uar_broadcast, _ = ak.broadcast_arrays(weight_m1_uar_per_event, energy_m1_uar)
    e_m1_uar_psd, w_m1_uar_psd = weighted_flat_energy(
        energy_m1_uar[psd_m1_uar], weight_m1_uar_broadcast[psd_m1_uar]
    )

    veto_label = f"NPE ≤ {npe_threshold}" if has_npe[iso] else f"LAr energy ≤ {lar_energy_threshold_kev} keV"

    fig, ax = plt.subplots(figsize=(10, 7))
    for data, weights, label, color in [
        (e_m1,         w_m1         / spec_bin_width, 'M1',                                                     'C0'),
        (e_m1_uar,     w_m1_uar     / spec_bin_width, f'M1 + AC UAr ({veto_label})',                            'C1'),
        (e_m1_uar_psd, w_m1_uar_psd / spec_bin_width, f'M1 + AC UAr + PSD (AoE_class > {aoe_class_threshold})', 'C3'),
    ]:
        if len(data) > 0:
            ax.hist(data, bins=spec_bins, weights=weights,
                    histtype='step', linewidth=2, label=label, color=color, alpha=0.85)

    ax.axvline(ROI_low,  color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(ROI_high, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvspan(ROI_low, ROI_high, alpha=0.08, color='red',
               label=f'ROI [{ROI_low}–{ROI_high} keV]')
    ax.set_xlabel('Energy [keV]')
    ax.set_ylabel('Rate [cts / keV / kg / yr]')
    ax.set_yscale('log')
    ax.set_xlim(spec_e_min, spec_e_max)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.legend(fontsize=12)
    ax.set_title(f'Normalized Ge spectrum — {iso} (1000 kg detector)')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ge_spectra_cuts_normalized_{iso}.png", dpi=150)
    plt.show()

In [ ]:
# %%
# Cell 15
# Background rate table by isotope and material region, in cts/keV/kg/yr
roi_width_keV = ROI_high - ROI_low
 
 
def weighted_rate_table(iso, sim, masks, spectra, region_name, weight_by_region):
    psd_pass      = masks["psd_pass"]
    m1_pass       = masks["m1_pass"]
    uar_veto_pass = masks["uar_veto_pass"]
    m1_uar_pass   = masks["m1_uar_pass"]
    psd_m1_uar    = spectra["psd_m1_uar"]
 
    for material, weight in weight_by_region.items():
        in_region = ak.Array(region_name == material)
 
        n_no_cut  = count_hits_in_roi(sim.energy[in_region], ROI_low, ROI_high)
        n_m1      = count_hits_in_roi(sim.energy[m1_pass & in_region], ROI_low, ROI_high)
        n_m1_uar  = count_hits_in_roi(sim.energy[m1_uar_pass & in_region], ROI_low, ROI_high)
 
        in_region_m1_uar = in_region[m1_uar_pass]
        n_m1_uar_psd = count_hits_in_roi(
            sim.energy[m1_uar_pass][in_region_m1_uar], ROI_low, ROI_high,
            mask=psd_m1_uar[in_region_m1_uar]
        )
 
        n_psd_only = count_hits_in_roi(sim.energy, ROI_low, ROI_high,
                                        mask=psd_pass & in_region)
 
        uar_pass_sim  = sim.energy[uar_veto_pass & in_region]
        n_uar_only    = count_hits_in_roi(uar_pass_sim, ROI_low, ROI_high)
 
        in_region_uar = in_region[uar_veto_pass]
        n_uar_psd = count_hits_in_roi(
            sim.energy[uar_veto_pass][in_region_uar], ROI_low, ROI_high,
            mask=psd_pass[uar_veto_pass][in_region_uar]
        )
 
        rows = [
            ("No cuts",           n_no_cut),
            ("PSD only",          n_psd_only),
            ("AC UAr only",       n_uar_only),
            ("M1 only",           n_m1),
            ("M1 + AC UAr",       n_m1_uar),
            ("AC UAr + PSD",      n_uar_psd),
            ("M1 + AC UAr + PSD", n_m1_uar_psd),
        ]
 
        print("=" * 80)
        print(f"{iso} / {material}  (scale factor {weight:.4e} cts/(yr·kg) per raw MC event)")
        print("=" * 80)
        print(f"{'Cut Scenario':<22} {'Raw hits':<10} {'Rate [cts/keV/kg/yr]':<22} {'σ [cts/keV/kg/yr]'}")
        print("-" * 80)
        for label, n_hits in rows:
            rate  = n_hits * weight / roi_width_keV
            sigma = np.sqrt(n_hits) * weight / roi_width_keV
            print(f"{label:<22} {n_hits:<10,} {rate:<22.4e} {sigma:.4e}")
        print()
 
 
for iso, sim in sims.items():
    vtx_z_iso   = ak.to_numpy(sim.vtx_z)
    region_name = assign_region(vtx_z_iso, material_regions, top_z)
    weighted_rate_table(iso, sim, masks_by_isotope[iso], spectra_by_isotope[iso],
                        region_name, scale_factor_by_isotope[iso])

In [ ]:
# %%
# Cell 16
# Precompute per-event ROI hit counts as flat numpy arrays.
# This is done once here so the scan loop (Cell 17) only does numpy
# operations -- no awkward array traversals inside the scan at all.
#
# n_roi_before[iso]: int array, shape (n_events,)
#   number of hits per event that fall in the ROI, no cuts applied
#
# n_roi_after[iso]:  int array, shape (n_events,)
#   number of hits per event in the ROI after M1 + AC UAr + PSD,
#   placed back into the full event index (0 for events that don't pass)
#
# vtx_z_np[iso]: float array, shape (n_events,)
#   vtx_z as a plain numpy array, aligned with the two arrays above
 
n_roi_before = {}
n_roi_after  = {}
vtx_z_np     = {}
 
for iso, sim in sims.items():
    vtx_z_np[iso] = ak.to_numpy(sim.vtx_z)
 
    # Before cuts: count hits in ROI per event
    in_roi_hits = (sim.energy >= ROI_low) & (sim.energy <= ROI_high)
    n_roi_before[iso] = ak.to_numpy(ak.sum(in_roi_hits, axis=-1)).astype(int)
 
    # After M1 + AC UAr + PSD: hits must also pass PSD and fall in ROI
    m           = masks_by_isotope[iso]
    m1_uar_pass = m["m1_uar_pass"]
    psd_m1_uar  = spectra_by_isotope[iso]["psd_m1_uar"]
 
    energy_m1_uar   = sim.energy[m1_uar_pass]
    in_roi_and_psd  = psd_m1_uar & ((energy_m1_uar >= ROI_low) & (energy_m1_uar <= ROI_high))
    hits_after      = ak.to_numpy(ak.sum(in_roi_and_psd, axis=-1)).astype(int)
 
    # Scatter back into a full-length array (0 for events not passing M1+UAr)
    m1_uar_idx              = np.where(ak.to_numpy(m1_uar_pass))[0]
    n_after_full            = np.zeros(len(sim), dtype=int)
    n_after_full[m1_uar_idx] = hits_after
    n_roi_after[iso]        = n_after_full
 
    print(f"{iso}: ROI hits before cuts = {n_roi_before[iso].sum():,}, "
          f"after all cuts = {n_roi_after[iso].sum():,}")

In [ ]:
 
# %%
# Cell 17
# 2D scan over [L1, L2] (EFCu and Cu thicknesses).
# Inner loop is pure numpy: no parquet reads, no awkward operations.
# Constraint L1 <= L2; cells where L1 > L2 are left as NaN.
# Volume per region is estimated proportionally from the pooled vertex
# z-distribution (equivalent to voxel method for uniform vertex density).
 
scan_n_points  = 100   # grid resolution per axis
L_values       = np.linspace(0, bottom_distance_m, scan_n_points)
n_pooled_total = len(pooled_vtx["z"])
 
roi_before   = np.full((scan_n_points, scan_n_points), np.nan)
roi_after    = np.full((scan_n_points, scan_n_points), np.nan)
sigma_before = np.full((scan_n_points, scan_n_points), np.nan)
sigma_after  = np.full((scan_n_points, scan_n_points), np.nan)
 
for i, L1 in enumerate(L_values):
    for j, L2 in enumerate(L_values):
        if L1 > L2:
            continue
 
        scan_regions = build_contiguous_regions(material_order, [L1, L2], bottom_distance_m)
        # Volume per region from pooled z-distribution
        V_R = {}
        for material, dist_start, dist_end in scan_regions:
            z_low, z_high = get_region_z_bounds(top_z, dist_start, dist_end)
            n_R = int(np.sum((pooled_vtx["z"] >= z_low) & (pooled_vtx["z"] < z_high)))
            V_R[material] = V_total * n_R / n_pooled_total
 
        rate_before   = 0.0
        rate_after    = 0.0
        sigma2_before = 0.0
        sigma2_after  = 0.0
 
        for iso in sims:
            for material, dist_start, dist_end in scan_regions:
                V_Rm = V_R[material]
                if V_Rm == 0:
                    continue
                sa = specific_activity_bq_per_kg[iso].get(material)
                if sa is None:
                    continue
 
                sf = (V_Rm * density_lookup[material] * sa * seconds_per_year
                      / (n_sim_total[iso] * V_Rm / V_total)
                      / Ge_total_mass_kg)
 
                z_low, z_high = get_region_z_bounds(top_z, dist_start, dist_end)
                in_R = (vtx_z_np[iso] >= z_low) & (vtx_z_np[iso] < z_high)
 
                N_before = n_roi_before[iso][in_R].sum()
                N_after  = n_roi_after[iso][in_R].sum()
                w = sf / roi_width_keV
 
                rate_before   += N_before * w
                rate_after    += N_after  * w
                sigma2_before += N_before * w**2  # Poisson: var(N*w) = N*w^2
                sigma2_after  += N_after  * w**2
 
        roi_before[i, j]   = rate_before
        roi_after[i, j]    = rate_after
        sigma_before[i, j] = np.sqrt(sigma2_before)
        sigma_after[i, j]  = np.sqrt(sigma2_after)
 
    print(f"  row {i+1}/{scan_n_points} done")
 
print("Scan complete.")
 

In [ ]:

# %%
# Cell 19
# Plot the 2D scan results
from matplotlib.colors import LogNorm
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
# Shared color scale — set manually or leave as None to auto-derive
vmin_manual = 1e-6   # set to None to auto-derive from data
vmax_manual = 1e-1   # set to None to auto-derive from data
 
all_vals = np.concatenate([
    roi_before[np.isfinite(roi_before) & (roi_before > 0)].flatten(),
    roi_after[np.isfinite(roi_after)   & (roi_after  > 0)].flatten(),
])
vmin = vmin_manual if vmin_manual is not None else (all_vals.min() if len(all_vals) else 1e-10)
vmax = vmax_manual if vmax_manual is not None else (all_vals.max() if len(all_vals) else 1.0)
 
for ax, data, title in [
    (axes[0], roi_before, "Before cuts"),
    (axes[1], roi_after,  "After all cuts (M1 + AC UAr + PSD)"),
]:
    data_plot = np.where(data > 0, data, np.nan)
 
    pcm = ax.pcolormesh(
        L_values, L_values, data_plot.T,
        norm=LogNorm(vmin=vmin, vmax=vmax),
        cmap="viridis", shading="auto"
    )
    plt.colorbar(pcm, ax=ax, label="Rate [cts / keV / kg / yr]")
 
    # Contour lines only on the after-cuts panel
    if data is roi_after:
        contour_levels = [1e-5, 5e-5, 1e-4, 2e-4]
        data_contour = np.where(np.isfinite(data_plot), data_plot, 0)
        cs = ax.contour(
            L_values, L_values, data_contour.T,
            levels=contour_levels,
            colors=["black", "black", "black"],
            linewidths=1.5,
        )
        #ax.clabel(cs, fmt={l: f"$10^{{{int(np.log10(l))}}}$" for l in contour_levels},
         #         fontsize=9, inline=True)
 
    ax.axvline(boundary_distances_m[0], color='red', linestyle='--', linewidth=1.2,
               label=f"L1 = {boundary_distances_m[0]} m")
    ax.axhline(boundary_distances_m[1], color='orange', linestyle='--', linewidth=1.2,
               label=f"L2 = {boundary_distances_m[1]} m")
    ax.plot(L_values, L_values, color='white', linewidth=1, linestyle=':',
            label="L1 = L2")
    ax.set_xlabel("L1: steel/Cu boundary from top [m]")
    ax.set_ylabel("L2: Cu/EFCu boundary from top [m]")
    ax.invert_yaxis()
    ax.set_title(f"ROI rate — {title}\n(summed Bi-214 + Tl-208)")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.4)
 
plt.tight_layout()
plt.savefig(f"{output_dir}/scan_roi_rate_2d.png", dpi=150)
plt.show()


In [ ]:

# %%
# Cell 19
# 1D slices through the scan at the standard boundary values.
# Plot 1: along x-axis (L1 varies, L2 fixed at boundary_distances_m[1])
# Plot 2: along y-axis (L2 varies, L1 fixed at boundary_distances_m[0])
# Both show rate before and after cuts with Poisson error bars.
 
# Find the grid indices closest to the two standard boundary values
i_std = int(np.argmin(np.abs(L_values - boundary_distances_m[0])))
j_std = int(np.argmin(np.abs(L_values - boundary_distances_m[1])))
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
 
# --- Plot 1: along x-axis, L2 fixed at standard value ---
x_vals      = L_values
y_before    = roi_before[:, j_std]
y_after     = roi_after[:, j_std]
ye_before   = sigma_before[:, j_std]
ye_after    = sigma_after[:, j_std]
 
# mask invalid points (L1 > L2 at this fixed L2)
valid = x_vals <= L_values[j_std]
 
ax1.errorbar(x_vals[valid], y_before[valid], yerr=ye_before[valid],
             fmt='o-', color='C0', capsize=4, label='Before cuts')
ax1.errorbar(x_vals[valid], y_after[valid],  yerr=ye_after[valid],
             fmt='s-', color='C3', capsize=4, label='After all cuts')
ax1.axvline(boundary_distances_m[0], color='gray', linestyle='--', linewidth=1.2,
            label=f"Standard L1 = {boundary_distances_m[0]} m")
ax1.set_xlabel("L1: steel/Cu boundary from top [m]")
ax1.set_ylabel("Rate [cts / keV / kg / yr]")
ax1.set_title(f"Scan along L1 at fixed L2 = {L_values[j_std]:.2f} m")
ax1.set_yscale('log')
ax1.set_xlim(0, bottom_distance_m)
ax1.legend(fontsize=11)
ax1.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
 
# --- Plot 2: along y-axis, L1 fixed at standard value ---
x_vals      = L_values
y_before    = roi_before[i_std, :]
y_after     = roi_after[i_std, :]
ye_before   = sigma_before[i_std, :]
ye_after    = sigma_after[i_std, :]
 
# mask invalid points (L2 < L1 at this fixed L1)
valid = x_vals >= L_values[i_std]
 
ax2.errorbar(x_vals[valid], y_before[valid], yerr=ye_before[valid],
             fmt='o-', color='C0', capsize=4, label='Before cuts')
ax2.errorbar(x_vals[valid], y_after[valid],  yerr=ye_after[valid],
             fmt='s-', color='C3', capsize=4, label='After all cuts')
ax2.axvline(boundary_distances_m[1], color='gray', linestyle='--', linewidth=1.2,
            label=f"Standard L2 = {boundary_distances_m[1]} m")
ax2.set_xlabel("L2: Cu/EFCu boundary from top [m]")
ax2.set_ylabel("Rate [cts / keV / kg / yr]")
ax2.set_title(f"Scan along L2 at fixed L1 = {L_values[i_std]:.2f} m")
ax2.set_yscale('log')
ax2.set_xlim(0, bottom_distance_m)
ax2.legend(fontsize=11)
ax2.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
 
plt.tight_layout()
plt.savefig(f"{output_dir}/scan_roi_1d_slices.png", dpi=150)
plt.show()


In [ ]:

# %%
# Cell 21
# vtx_z origin for events depositing energy near 2614 keV.
# Left: raw counts. Right: normalized by hollow cylinder volume (counts/cm³).
peak_center   = 2614   # keV
peak_window   = 10     # keV, +/- around peak center
z_bin_width   = 0.05   # meters
 
z_bins_peak = np.arange(
    min(ak.to_numpy(sims[iso].vtx_z).min() for iso in sims),
    max(ak.to_numpy(sims[iso].vtx_z).max() for iso in sims) + z_bin_width,
    z_bin_width
)
z_bin_centers = 0.5 * (z_bins_peak[:-1] + z_bins_peak[1:])
 
vtx_z_all = np.concatenate([
    ak.to_numpy(sim.vtx_z[ak.any(
        (sim.energy >= peak_center - peak_window) &
        (sim.energy <= peak_center + peak_window),
        axis=-1
    )])
    for sim in sims.values()
])
 
counts, _ = np.histogram(vtx_z_all, bins=z_bins_peak)
 
# Volume per bin from Cell 9 r_inner/r_outer
r_inner_interp = np.interp(z_bin_centers, z_centers, r_inner,
                            left=np.nan, right=np.nan)
r_outer_interp = np.interp(z_bin_centers, z_centers, r_outer,
                            left=np.nan, right=np.nan)
V_slice_cm3 = np.pi * (r_outer_interp**2 - r_inner_interp**2) * z_bin_width * 1e6
density         = np.where(V_slice_cm3 > 0, counts / V_slice_cm3, np.nan)
 
_top_z = max(ak.to_numpy(sims[iso].vtx_z).max() for iso in sims)
 
def _add_boundaries(ax):
    for i, (dist, label) in enumerate(zip(
        boundary_distances_m,
        [f"{material_order[0]}/{material_order[1]}",
         f"{material_order[1]}/{material_order[2]}"]
    )):
        ax.axvline(_top_z - dist, color=f"C{i+2}", linestyle='--',
                   linewidth=1.2, label=label)
 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
bin_label = f"{int(z_bin_width*100)} cm bins"
 
# Top-left: raw counts
axes[0, 0].step(z_bins_peak[:-1], counts, where='post',
                linewidth=1.5, color='C0', label=f"n = {len(vtx_z_all):,}")
_add_boundaries(axes[0, 0])
axes[0, 0].set_xlabel("vtx_z [m]")
axes[0, 0].set_ylabel("Counts")
axes[0, 0].set_title(f"Raw counts — {peak_center} ± {peak_window} keV  ({bin_label})")
axes[0, 0].set_yscale('log')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.5)
 
# Top-right: counts / cm³
axes[0, 1].step(z_bins_peak[:-1], density, where='post',
                linewidth=1.5, color='C1', label=f"n = {len(vtx_z_all):,}")
_add_boundaries(axes[0, 1])
axes[0, 1].set_xlabel("vtx_z [m]")
axes[0, 1].set_ylabel("Counts / cm³")
axes[0, 1].set_title(f"Volume-normalized counts — {peak_center} ± {peak_window} keV  ({bin_label})")
axes[0, 1].set_yscale('log')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.5)
 
# Bottom row: 2D histogram of LAr energy vs vtx_z
# x: vtx_z (origin z), y: total LAr energy per event
 
# Collect vtx_z and total LAr energy for peak events across isotopes
vtx_z_2d  = np.concatenate([
    ak.to_numpy(sim.vtx_z[ak.any(
        (sim.energy >= peak_center - peak_window) &
        (sim.energy <= peak_center + peak_window),
        axis=-1
    )])
    for sim in sims.values()
])
lar_2d = np.concatenate([
    ak.to_numpy(ak.sum(sim.lar_energy[ak.any(
        (sim.energy >= peak_center - peak_window) &
        (sim.energy <= peak_center + peak_window),
        axis=-1
    )], axis=-1))
    for sim in sims.values()
])
 
# Log-spaced y-bins for LAr energy (skip zero)
lar_positive = lar_2d[lar_2d > 0]
lar_bins = np.logspace(np.log10(lar_positive.min()), np.log10(lar_positive.max()), 80)
 
H, xedges, yedges = np.histogram2d(vtx_z_2d, lar_2d,
                                    bins=[z_bins_peak, lar_bins])
 
# Volume-normalize each z-column by wall volume
H_norm = H.copy().astype(float)
for k, vcm3 in enumerate(V_slice_cm3):
    if np.isfinite(vcm3) and vcm3 > 0:
        H_norm[k, :] /= vcm3
    else:
        H_norm[k, :] = np.nan
 
for ax, data, ylabel, title in [
    (axes[1, 0], H,      "LAr energy [keV]",        "LAr energy vs vtx_z — raw counts"),
    (axes[1, 1], H_norm, "LAr energy [keV]",        "LAr energy vs vtx_z — counts / cm³"),
]:
    data_plot = np.where(data > 0, data, np.nan)
    pcm = ax.pcolormesh(xedges, yedges, data_plot.T,
                        norm=plt.matplotlib.colors.LogNorm(), cmap="viridis",
                        shading="auto")
    plt.colorbar(pcm, ax=ax, label="Counts" if data is H else "Counts / cm³")
    _add_boundaries(ax)
    ax.set_xlabel("vtx_z [m]")
    ax.set_ylabel(ylabel)
    #ax.set_yscale('log')
    ax.set_title(f"{title} ({bin_label})")
    ax.legend(fontsize=9)
    ax.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.3)
 
plt.tight_layout()
plt.savefig(f"{output_dir}/vtxz_2614keV.png", dpi=150)
plt.show()
plt.tight_layout()
plt.savefig(f"{output_dir}/vtxz_2614keV.png", dpi=150)
plt.show()
 
